In [1]:
import numpy as np
import pandas as pd
from types import resolve_bases
import pickle
import plotly.express as px
from SamplingMethods import Sampler_class
from ax.api.client import Client
from ax.api.configs import RangeParameterConfig
from ax.generation_strategy.center_generation_node import CenterGenerationNode
from ax.generation_strategy.transition_criterion import MinTrials
from ax.generation_strategy.generation_strategy import GenerationStrategy
from ax.generation_strategy.generation_node import GenerationNode
from ax.generation_strategy.model_spec import GeneratorSpec
from ax.modelbridge.registry import Generators
from gpytorch.kernels import MaternKernel
from botorch.models import SingleTaskGP
from botorch.models.transforms.input import Warp
from botorch.models.map_saas import AdditiveMapSaasSingleTaskGP
from ax.utils.stats.model_fit_stats import MSE
from ax.models.torch.botorch_modular.surrogate import SurrogateSpec, ModelConfig
from botorch.acquisition.analytic import ProbabilityOfImprovement
import copy

In [2]:
client = Client()
gp_model = client.load_from_json_file("/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/StoichModelGP/ModelGP_M15.json")
gp_model.get_next_trials(max_trials=1)

{56: {'n_ci': 0.7773318911025692, 'n_it': 0.3933436206813303}}

In [3]:
def SurrogateModelOfReality(n_ci,n_it):
    y_pred = gp_model.predict([{"n_ci":n_ci,"n_it":n_it}])[0]["t1"][0]
    return np.float64(y_pred)

In [4]:
def PredictorsToCaStoichs(s1,b1):
    return (0.5+(2.0-0.5)*s1)*b1

def PredictorsToIaStoichs(s2,b1):
    return (1.0+(2.0-1.0)*s2)*(1-b1)

In [5]:
y_max_lis = []

for i in range(100):
    client = Client()
    parameters = [
        RangeParameterConfig(name="s1", parameter_type="float", bounds=tuple([0, 1])),
        RangeParameterConfig(name="s2", parameter_type="float", bounds=tuple([0, 1])),
        RangeParameterConfig(name="b1", parameter_type="float", bounds=tuple([0, 1])),
    ]
    client.configure_experiment(parameters=parameters)
    def construct_generation_strategy(
        generator_spec: GeneratorSpec, node_name: str,
    ) -> GenerationStrategy:
        """Constructs a Center + Sobol + Modular BoTorch `GenerationStrategy`
        using the provided `generator_spec` for the Modular BoTorch node.
        """
        botorch_node = GenerationNode(
            node_name=node_name,
            model_specs=[generator_spec],
        )
        return GenerationStrategy(
            name=f"{node_name}",
            nodes=[botorch_node]
        )

    # Let's construct the simplest version with all defaults.
    construct_generation_strategy(
        generator_spec=GeneratorSpec(model_enum=Generators.BOTORCH_MODULAR),
        node_name="Modular BoTorch",
    )

    surrogate_spec = SurrogateSpec(
        model_configs=[
            # Select between two models:
            # An additive mixture of relatively strong SAAS priors with input Warping.
            # A relatively vanilla GP with a Matern kernel.
            ModelConfig(
                botorch_model_class=SingleTaskGP,
                covar_module_class=MaternKernel,
                covar_module_options={"nu": 2.5},
            ),
        ],
        eval_criterion=MSE,  # Select the model to use as the one that minimizes mean squared error.
        allow_batched_models=False,  # Forces each metric to be modeled with an independent BoTorch model.
        # If we wanted to specify different options for different metrics.
        # metric_to_model_configs: dict[str, list[ModelConfig]]
    )

    generator_spec = GeneratorSpec(
        model_enum=Generators.BOTORCH_MODULAR,
        model_kwargs={
            "surrogate_spec": surrogate_spec,
            "botorch_acqf_class": ProbabilityOfImprovement,
            # Can be used for additional inputs that are not constructed
            # by default in Ax. We will demonstrate below.
            "acquisition_options": {},
        },
        # We can specify various options for the optimizer here.
        model_gen_kwargs = {
            "model_gen_options": {
                "optimizer_kwargs": {
                    "num_restarts": 20,
                    "sequential": False,
                    "options": {
                        "batch_limit": 5,
                        "maxiter": 200,
                    },
                },
            },
        }
    )

    generation_strategy = construct_generation_strategy(
        generator_spec=generator_spec,
        node_name="BoTorch w/ Model Selection",
    )
    generation_strategy

    client.set_generation_strategy(
        generation_strategy=generation_strategy,
    )

    metric_name = "t1" # this name is used during the optimization loop in Step 5
    objective = f"{metric_name}" # minimization is specified by the negative sign

    client.configure_optimization(objective=objective)

    # Quasirandom Sampling Exercise
    sampler = Sampler_class()
    Parameters_lis = [
        {"name":"s1", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"s2", "type":"range","bounds":[0,1],"value_type":"float"},
        {"name":"b1", "type":"range","bounds":[0,1],"value_type":"float"}
    ]
    X = sampler.three.QuasirandomSampler3D_func(8,Parameters_lis).T

    for array in X:
        s1 = array[0]
        s2 = array[1]
        b1 = array[2]
        n_ci = PredictorsToCaStoichs(s1,b1)
        n_it = PredictorsToIaStoichs(s2,b1)
        my_parameters = {"s1": s1, "s2": s2, "b1": b1}
        trial_index = client.attach_trial(parameters=my_parameters)
        client.complete_trial(trial_index=trial_index,raw_data={"t1": SurrogateModelOfReality(n_ci,n_it)})

    for _ in range(7):
        IterationClient = copy.deepcopy(client)
        IterationTrials = {}
        for __ in range(3):
            SampleTrial = IterationClient.get_next_trials(max_trials=1)
            for trial_index, parameters in SampleTrial.items():
                IterationTrials[trial_index]=parameters
                s1 = parameters["s1"]
                s2 = parameters["s2"]
                b1 = parameters["b1"]
                result = IterationClient.predict([{"s1":s1,"s2":s2,"b1":b1}])[0]["t1"][0]
                raw_data = {metric_name: result}
                IterationClient.complete_trial(trial_index=trial_index, raw_data=raw_data)
        for trial_index, parameters in IterationTrials.items():
            client.attach_trial(parameters=parameters)
            s1 = parameters["s1"]
            s2 = parameters["s2"]
            b1 = parameters["b1"]
            n_ci = PredictorsToCaStoichs(s1,b1)
            n_it = PredictorsToIaStoichs(s2,b1)
            result = SurrogateModelOfReality(n_ci,n_it)
            raw_data = {metric_name: result}
            client.complete_trial(trial_index=trial_index, raw_data=raw_data)

    print(f"Trial {i} =========================================")
    y_max = np.max(np.array(np.array(client.summarize().t1).tolist()[0:27]))
    print(y_max)
    y_max_lis.append(y_max)
    print()

y_max_arr = np.array(y_max_lis)
print(y_max_arr)

Trial 0 =========================================
13.710006879117081

Trial 1 =========================================
16.5611646340553

Trial 2 =========================================
18.181614368226555

Trial 3 =========================================
16.788752253269045

Trial 4 =========================================
13.881495897495066

Trial 5 =========================================
17.83062585106646

Trial 6 =========================================
16.705072790862754

Trial 7 =========================================
13.87918394790682

Trial 8 =========================================
13.877530521181086

Trial 9 =========================================
17.533842009895388

Trial 10 =========================================
13.830797188419766

Trial 11 =========================================
13.862985560359084

Trial 12 =========================================
13.359867352254637

Trial 13 =========================================
17.00554395958364

Trial 14 ============

/Users/thomasdodd/miniconda3/envs/ax1_env/lib/python3.13/site-packages/botorch/fit.py:215: OptimizationWarning: `scipy_minimize` terminated with status OptimizationStatus.FAILURE, displaying original message from `scipy.optimize.minimize`: ABNORMAL: 
  result = optimizer(mll, closure=closure, **optimizer_kwargs)


Trial 80 =========================================
13.574095018551972

Trial 81 =========================================
17.735800631426127

Trial 82 =========================================
13.69267431974129

Trial 83 =========================================
13.718071778221372

Trial 84 =========================================
15.595867731009896

Trial 85 =========================================
18.00514808783082

Trial 86 =========================================
13.711711158584876

Trial 87 =========================================
17.387704934268555

Trial 88 =========================================
13.56192883149607

Trial 89 =========================================
13.868372716288661

Trial 90 =========================================
18.182318467100053

Trial 91 =========================================
13.877953915318141

Trial 92 =========================================
13.669237822212

Trial 93 =========================================
17.55674095853442

Trial 94 ====

In [6]:
print(f"Max = {np.max(y_max_arr)}")
print(f"Avg = {np.average(y_max_arr)}")
print(f"Std = {np.std(y_max_arr)}")

Max = 18.272600746248287
Avg = 15.295107897322307
Std = 1.9376496368470952


In [7]:
print(y_max_arr.tolist())

[13.710006879117081, 16.5611646340553, 18.181614368226555, 16.788752253269045, 13.881495897495066, 17.83062585106646, 16.705072790862754, 13.87918394790682, 13.877530521181086, 17.533842009895388, 13.830797188419766, 13.862985560359084, 13.359867352254637, 17.00554395958364, 13.863629373284144, 17.89635673914382, 17.691270658244672, 13.3121351076842, 17.903195586531737, 18.24098740589219, 18.218709798678688, 13.019410983791603, 18.266537078945124, 13.776876977127191, 17.955606626104238, 14.259123752951997, 18.254788270602987, 15.893616762179452, 13.778879387870816, 13.64134381811581, 13.361741922000771, 13.854196601743975, 17.421008912876754, 13.880081466018606, 17.135223290050792, 13.850963763524222, 16.5875080537738, 13.827861972261516, 13.833743608662742, 13.84923027065634, 13.732243950735507, 13.842297319726477, 13.877100578918824, 16.800386975980242, 17.888977479688485, 18.272600746248287, 16.68479279491694, 13.564064218334796, 13.064798341096678, 13.644234517392643, 13.8133798377

In [8]:
# filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/SequentialTestswGPModel/DataGenerated/normal_PI_9_27_3.pkl"
# latestdf = pd.DataFrame(y_max_arr)
# pd.to_pickle(obj=latestdf,filepath_or_buffer=filepath)

In [9]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M15/DataGenerated/normal_PI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
latestdf = pd.DataFrame(y_max_arr)
newdf = pd.concat(objs=[loadeddf,latestdf],axis=0)
newdf = newdf.reset_index(drop=True)
pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)

In [10]:
filepath = "/Users/thomasdodd/Library/CloudStorage/OneDrive-MillfieldEnterprisesLimited/Cambridge/PhD/writing/papers/UoC_Paper1/sandbox/Emulation/TestStoichModelGP_M15/DataGenerated/normal_PI_9_27_3.pkl"
loadeddf = pd.read_pickle(filepath_or_buffer=filepath)
print(loadeddf)
# newdf = loadeddf.drop(loadeddf.index, inplace=True)
# pd.to_pickle(obj=newdf,filepath_or_buffer=filepath)
# print(newdf)

             0
0    18.200230
1    14.253545
2    18.220578
3    17.918699
4    18.272490
..         ...
995  17.876096
996  13.659763
997  13.867412
998  13.839504
999  15.642523

[1000 rows x 1 columns]
